## Needed dependencies:
- jupyter
- q-alchemy-sdk-py
- python-dotenv
- numpy
- qiskit

In [2]:
from qiskit.quantum_info import Statevector
from qiskit import QuantumCircuit
from q_alchemy.qiskit_integration import qiskit_batch_initialize
import numpy as np
from q_alchemy.initialize import OptParams
from dotenv import load_dotenv
import os

load_dotenv("../.env")

True

## Create 4 random states

In [3]:
n_qubits = 4
n_states = 4
state_vectors = [np.random.rand(2 ** n_qubits) + np.random.rand(2 ** n_qubits) * 1j for i in range (n_states)]
state_vectors = [sv / np.linalg.norm(sv) for sv in state_vectors]

## Submit batch job to Q-Alchemy

`gate_list` is the list of preparation circuits produced by Q-Alchemy.

In [4]:

max_fidelity_loss = 0.05

gate_list = qiskit_batch_initialize(
    state_vectors=state_vectors,
    opt_params=OptParams(
        max_fidelity_loss=max_fidelity_loss,
        api_key=os.environ["Q_ALCHEMY_API_KEY"]
    )
)

C:\Users\KeithNg\Repos\qalchemy\q-alchemy-sdk-py\.venv\Lib\site-packages\pinexq\client\job_management\enterjma.py:90: UserWarning: Version mismatch between 'pinexq_client' (v9.5.0) and 'JobManagementAPI' (v9.3.0)! 
  warnings.warn(


## Test Q-Alchemy preparation gates on local SV simulator

In [5]:

qcs = [QuantumCircuit(n_qubits) for gate in gate_list]
for gate, qc in zip(gate_list, qcs):
    qc.append(gate, range(n_qubits))

qiskit_states = [Statevector(circuit).data for circuit in qcs]

for init_state, qiskit_state in zip(state_vectors, qiskit_states):
    fidelity = abs(np.vdot(init_state, qiskit_state))**2
    print(fidelity)
    diff_norm = np.linalg.norm(init_state - qiskit_state)**2
    print(diff_norm)

0.9714249294617396
0.02878217392218169
0.9547078194736118
0.04581697942735006
0.9839195465749169
0.016145623716381015
0.977571872534036
0.02255531300212053


## Draw a preparation gate

In [6]:
print(qcs[0])
print(qcs[0].decompose())

     ┌───────┐
q_0: ┤0      ├
     │       │
q_1: ┤1      ├
     │  QAl0 │
q_2: ┤2      ├
     │       │
q_3: ┤3      ├
     └───────┘
global phase: 5.0654
     ┌─────────────────────────┐                                 ┌───┐»
q_0: ┤ U(2.778,2.8997,-2.5145) ├─────────────────────────────────┤ X ├»
     └─────────────────────────┘┌───┐┌──────────────────────────┐└─┬─┘»
q_1: ───────────────────────────┤ X ├┤ U(1.4317,-1.3663,-1.281) ├──■──»
     ┌─────────────────────────┐└─┬─┘└──────────────────────────┘┌───┐»
q_2: ┤ U(1.3157,1.0427,1.5167) ├──┼──────────────────────────────┤ X ├»
     └────┬────────────────┬───┘  │  ┌──────────────────────────┐└─┬─┘»
q_3: ─────┤ U(0.66467,0,0) ├──────■──┤ U(1.4456,-2.4957,1.4298) ├──■──»
          └────────────────┘         └──────────────────────────┘     »
«     ┌─────────────────────────────┐┌───┐┌─────────────────────────┐┌───┐»
«q_0: ┤ U(1.5523,-0.15096,-0.12075) ├┤ X ├┤ U(1.4205,3.1184,-1.417) ├┤ X ├»
«     └────┬────────────────────┬───┘└─┬─┘└─